# Lab W7: Adopsi Repo Eksternal

## Pre-flight Checklist

> [!IMPORTANT]
> Konsep yang ditandai (§) merujuk ke `07_W7_Text_Transformers_Repo_Adoption.md` §3 dan §D1-D7 (Pendalaman Repo Adoption).

**Yang Anda butuhkan sebelum mulai:**
- Bab W7 sudah dibaca, terutama §3 (urutan baca, repo_map.md, modifikasi minimal-invasif) dan §D2 Pendalaman (urutan luar-ke-dalam, smoke test 3-level, navigasi cepat dengan grep).
- Familiar dengan `git clone`, `git status`, `git diff`, `grep` atau `rg`.
- Lab W2 selesai (familiar dengan `template_repo` `src/` structure).

**Yang akan Anda hasilkan di akhir lab:**
- Repo eksternal di-clone (target: `pytorch/vision`).
- `repo_map.md` berisi 6 bagian: tujuan, struktur, entry point, model, data, config.
- 3 level smoke test pada kode eksternal: L1 import, L2 forward dengan dummy, L3 satu iterasi training.
- 1 komponen di-port secara minimal-invasif ke `template_repo/src/` (file baru atau argumen opsional).
- Baseline training + variasi dengan komponen yang di-port (2 kondisi x 2 seeds).
- Commit history kecil dan bermakna (4+ commits terpisah).

**Resource:**
- **Hardware:** CPU cukup untuk smoke test. GPU opsional untuk baseline comparison.
- **Storage:** ~200 MB (pytorch/vision shallow clone).
- **Estimasi waktu kerja:** 3-5 jam.

**Pendamping:** Bab W7 di `07_W7_Text_Transformers_Repo_Adoption.md`.

> [!TIP]
> Jangan tergoda untuk melewati bagian `repo_map.md` karena "sudah paham". Pemetaan tertulis adalah produk utama lab ini, bukan kode yang di-port.

## 1. Clone Repo Eksternal

In [ ]:
import os, sys
_root = os.path.abspath('..')
if _root not in sys.path:
    sys.path.insert(0, _root)

# Clone pytorch/vision (shallow clone untuk menghemat bandwidth)
external_dir = os.path.join(_root, 'external')
os.makedirs(external_dir, exist_ok=True)

if not os.path.exists(os.path.join(external_dir, 'vision')):
    !git clone --depth 1 https://github.com/pytorch/vision "{external_dir}/vision"
    print('Repo cloned.')
else:
    print('Repo already cloned. Pulling latest...')
    !git -C "{external_dir}/vision" pull

## 2. Eksplorasi Struktur: 15-Minute Map

Jelajahi struktur direktori `references/classification/` — ini adalah reference implementation untuk klasifikasi gambar di torchvision. Catat file-file utama dan peran masing-masing.

In [ ]:
vision_dir = os.path.join(external_dir, 'vision')
ref_dir = os.path.join(vision_dir, 'references', 'classification')

print('=== references/classification/ ===')
for f in sorted(os.listdir(ref_dir)):
    fpath = os.path.join(ref_dir, f)
    size = os.path.getsize(fpath) if os.path.isfile(fpath) else '-'
    print(f'  {f:30s} ({size} bytes)' if isinstance(size, int) else f'  {f:30s}/')

print()
print('=== File kunci dan perannya ===')
key_files = {
    'train.py': 'Entry point — argparse, main(), training loop. Cari `def main(args)` untuk memahami alur eksekusi.',
    'presets.py': 'Classification presets — augmentasi + transform yang dipreset. Pola yang umum di torchvision.',
    'transforms.py': 'TrivialAugmentWide, RandAugment, MixUp — augmentasi canggih. Calon kuat untuk di-port.',
    'utils.py': 'Distributed helpers, metric loggers, SmoothedValue. Utilitas yang diperlukan training loop.',
    'sampler.py': 'RASampler (Repeated Augmentation Sampler) — sampling strategy.',
}
for fname, desc in key_files.items():
    fpath = os.path.join(ref_dir, fname)
    if os.path.exists(fpath):
        lines = len(open(fpath, encoding='utf-8').readlines())
        print(f'  {fname:25s} ({lines:4d} lines) — {desc}')
    else:
        print(f'  {fname:25s} (MISSING) — {desc}')

In [ ]:
# Quick grep: temukan entry point main()
print('=== Mencari def main(args) di train.py ===')
with open(os.path.join(ref_dir, 'train.py'), encoding='utf-8') as f:
    for i, line in enumerate(f, 1):
        if 'def main' in line or 'def train_one_epoch' in line:
            print(f'  Line {i:4d}: {line.rstrip()}')

## 3. Tulis `repo_map.md`

Isi template 6-bagian di bawah. Ini adalah produk utama lab ini. Jangan copy-paste dari komentar kode — tulis dengan kata-kata sendiri setelah benar-benar membaca file yang dimaksud.

In [ ]:
repo_map_content = """
# repo_map.md — pytorch/vision (references/classification/)

## 1. Tujuan Repo
\n[Tulis: apa yang dilakukan repo ini? Klasifikasi gambar dengan model-model torchvision. Apa bedanya dari training loop sederhana di template_repo?]

## 2. Struktur Key Directory
\n[Tulis: struktur direktori references/classification/ dengan anotasi pendek per file]

## 3. Entry Point & Alur Eksekusi
\n[Tulis: dari `python train.py --model resnet18 ...`, apa yang terjadi? Bagaimana argumen diparse, model dibuat, data diload?]

## 4. Cara Model Didefinisikan
\n[Tulis: dari mana model diambil? torchvision.models. Bagaimana model dimodifikasi untuk klasifikasi kustom?]

## 5. Cara Data Masuk
\n[Tulis: dataset apa? Transformasi apa? Bagaimana ImageFolder digunakan? Apa peran presets?]

## 6. Config & Hyperparameter
\n[Tulis: bagaimana hyperparameter diset? argparse. Parameter kunci: lr, batch_size, epochs, model, dll.]
"""

map_path = os.path.join(_root, 'docs', 'repo_map_vision.md')
os.makedirs(os.path.dirname(map_path), exist_ok=True)
with open(map_path, 'w', encoding='utf-8') as f:
    f.write(repo_map_content)
print(f'repo_map.md template written to {map_path}')
print('Isi setiap section [Tulis: ...] dengan observasi Anda sendiri.')

## 4. Smoke Test 3-Level pada Kode Eksternal

Verifikasi bahwa kode eksternal benar-benar berfungsi sebelum kita memodifikasinya.

In [ ]:
import torch
import torch.nn as nn

# Level 1: Import — apakah model tersedia?
print('=== Smoke Test Level 1: Import ===')
import torchvision
from torchvision.models import resnet18, ResNet18_Weights
print(f'torchvision version: {torchvision.__version__}')

# Level 2: Dummy forward — apakah shape cocok?
print('\n=== Smoke Test Level 2: Dummy Forward ===')
model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
model.eval()
dummy = torch.randn(2, 3, 224, 224)
with torch.no_grad():
    out = model(dummy)
print(f'  Input: {dummy.shape}')
print(f'  Output: {out.shape}  (expected: (2, 1000))')
assert out.shape == (2, 1000), f'Shape mismatch: {out.shape}'
print('  ✓ Level 2 passed')

# Level 3: Satu iterasi training — apakah loss bisa dihitung?
print('\n=== Smoke Test Level 3: Satu Iterasi Training ===')
model.train()
opt = torch.optim.SGD(model.parameters(), lr=0.001)
crit = nn.CrossEntropyLoss()
labels = torch.randint(0, 1000, (2,))
opt.zero_grad()
loss = crit(model(dummy), labels)
loss.backward()
opt.step()
print(f'  Loss: {loss.item():.4f}')
print('  ✓ Level 3 passed — gradient mengalir, optimizer step berhasil')

## 5. Port Satu Komponen: Tambahkan MixUp ke `template_repo`

Kita akan mem-port MixUp dari `references/classification/transforms.py` ke `template_repo/src/augment.py` (file baru). MixUp adalah teknik augmentasi yang memadukan dua sampel secara linear.

In [ ]:
# Definisikan MixUp sederhana
class MixUp:
    """MixUp augmentation: x = alpha * x1 + (1-alpha) * x2, y = alpha * y1 + (1-alpha) * y2.
    Diadaptasi dari torchvision/references/classification/transforms.py.
    """
    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def __call__(self, x1, y1, x2, y2):
        lam = float(torch.distributions.Beta(self.alpha, self.alpha).sample()) if self.alpha > 0 else 1.0
        x = lam * x1 + (1 - lam) * x2
        return x, y1, y2, lam

# Smoke test MixUp
mixup = MixUp(alpha=1.0)
x1 = torch.randn(4, 3, 32, 32)
x2 = torch.randn(4, 3, 32, 32)
y1 = torch.randint(0, 10, (4,))
y2 = torch.randint(0, 10, (4,))
x_mixed, y1_m, y2_m, lam = mixup(x1, y1, x2, y2)
print(f'  Mixed x shape: {x_mixed.shape}')
print(f'  Lambda: {lam:.3f}')
print(f'  y1: {y1_m.tolist()}')
print(f'  y2: {y2_m.tolist()}')

In [ ]:
# Tulis ke file src/augment.py
augment_path = os.path.join(_root, 'src', 'augment.py')
with open(augment_path, 'w', encoding='utf-8') as f:
    f.write('''"""Augmentasi tambahan — port dari torchvision/references/classification/."""

import torch


class MixUp:
    """MixUp augmentation: x = lam*x1 + (1-lam)*x2.

    Diadaptasi dari torchvision references/classification/transforms.py.
    Mixup: Beyond Empirical Risk Minimization (Zhang et al., 2018, arXiv:1710.09412).
    """

    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def __call__(self, x1, y1, x2, y2):
        if self.alpha > 0:
            lam = float(torch.distributions.Beta(self.alpha, self.alpha).sample())
        else:
            lam = 1.0
        x = lam * x1 + (1 - lam) * x2
        return x, y1, y2, lam
''')
print(f'Port selesai: {augment_path}')

## 6. Baseline Training + Variasi (2 Kondisi × 2 Seeds)

Bandingkan training tanpa MixUp vs dengan MixUp, masing-masing 2 seeds. Hasilnya dicatat dalam tabel ringkasan.

In [ ]:
import subprocess

def run_training(config, seed, label):
    print(f'\n{"="*60}')
    print(f'Training: {label} — seed={seed}')
    print(f'{"="*60}')
    result = subprocess.run(
        [sys.executable, '-m', 'src.train', '--config', config, '--seed', str(seed)],
        capture_output=True, text=True, cwd=_root
    )
    if result.returncode != 0:
        print('STDERR:', result.stderr[-1000:])
    else:
        # Parse val_acc terakhir dari output
        for line in result.stdout.splitlines():
            if 'val_acc' in line:
                last_val = line
        print(last_val if 'last_val' in dir() else result.stdout[-500:])
    return result.returncode

print('Catatan: jalankan perintah berikut di terminal (bukan dalam notebook) untuk training penuh:')
print('  # Kondisi 1: Baseline (tanpa MixUp)')
print('  python -m src.train --config configs/baseline.yaml --seed 42')
print('  python -m src.train --config configs/baseline.yaml --seed 1337')
print()
print('  # Kondisi 2: Dengan MixUp (perlu modifikasi train.py untuk memanggil MixUp)')
print('  python -m src.train --config configs/baseline.yaml --seed 42  # dengan MixUp enabled')
print('  python -m src.train --config configs/baseline.yaml --seed 1337  # dengan MixUp enabled')
print()
print('Setelah training selesai, isi tabel di bawah:')

In [ ]:
# Tabel hasil — isi setelah training selesai
import json
from pathlib import Path

experiments_dir = Path(_root) / 'experiments'

rows = []
for exp_name in ['baseline_seed42', 'baseline_seed1337']:
    exp_dir = experiments_dir / exp_name
    if exp_dir.exists():
        summary = json.loads((exp_dir / 'summary.json').read_text())
        rows.append({
            'Kondisi': 'Baseline (no MixUp)',
            'Seed': exp_name.split('seed')[1],
            'Val Acc': summary.get('val_acc', 'N/A'),
            'Val Loss': summary.get('val_loss', 'N/A'),
        })

if rows:
    import pandas as pd
    print(pd.DataFrame(rows).to_string(index=False))
else:
    print('Belum ada hasil. Jalankan training dulu (lihat cell di atas).')

## 7. Git Workflow: 4+ Commits Terpisah

Commit history kecil dan bermakna. Jalankan dari terminal di folder `template_repo/`:

```bash
# Commit 1: Struktur direktori
mkdir -p external docs
echo 'external/' >> .gitignore
git add .gitignore
git commit -m "chore: add external/ to .gitignore"

# Commit 2: Clone repo eksternal
git clone --depth 1 https://github.com/pytorch/vision external/vision
# (external/ tidak di-track — hanya lokal)

# Commit 3: repo_map.md
git add docs/repo_map_vision.md
git commit -m "docs: add repo_map.md for pytorch/vision reference impl"

# Commit 4: Port MixUp ke src/
git add src/augment.py
git commit -m "feat: port MixUp augmentation from torchvision references"

# Commit 5 (opsional): Baseline comparison results
git add experiments/
git commit -m "exp: add MixUp vs baseline comparison (2 seeds)"
```

Pastikan setiap commit punya scope yang jelas dan pesan informatif.

## 8. Draft PR Description

Tulis deskripsi Pull Request jika komponen yang di-port akan di-submit ke upstream `template_repo`. Template:

### Draft PR: Add MixUp Augmentation

**Summary:**

> *[Tulis 2-3 kalimat: apa yang ditambahkan, dari mana sumbernya]*

**Changes:**
- `src/augment.py` (new): MixUp class adapted from torchvision `references/classification/transforms.py`

**Motivation:**

> *[Tulis: kenapa MixUp berguna? Apa bukti dari paper atau eksperimen?]*

**Testing:**
- Smoke test: MixUp pada dummy batch (4, 3, 32, 32) — output shape verified
- Baseline comparison: CIFAR-10 SimpleCNN, 2 seeds per condition

**Results (if available):**

| Condition | Seed | Val Acc |
| --- | --- | --- |
| Baseline | 42 | ... |
| Baseline | 1337 | ... |
| + MixUp | 42 | ... |
| + MixUp | 1337 | ... |

## 9. Refleksi

Jawab di sel di bawah:

1. **Berapa lama total: clone → map → port?** Apakah < 3 jam seperti target §D3 W7?

2. **Apa satu pola kode dari repo asing yang Anda adopsi** ke gaya penulisan sendiri? (contoh: cara mengorganisir argumen, penamaan, struktur class, dll.)

3. **Apa bagian yang TIDAK Anda port**, meskipun menarik — kenapa? (scope, dependency, kompleksitas)

4. **Apa yang paling mengejutkan** tentang struktur kode di pytorch/vision dibanding template_repo?

### Jawaban Refleksi

**1. Durasi total:**

> *[tulis di sini]*

**2. Pola kode yang diadopsi:**

> *[tulis di sini]*

**3. Bagian yang tidak di-port dan alasannya:**

> *[tulis di sini]*

**4. Hal paling mengejutkan:**

> *[tulis di sini]*